# 배경 생성 PoC — sd-turbo
그라데이션(임시 배경)을 AI 생성 배경으로 교체하기 위한 실험.
프롬프트 → 배경 생성 → 누끼·문구와 합성까지 확인한다.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16
).to("cuda")

In [ ]:
prompt = "warm cozy cafe interior, wooden table, soft morning light, blurred background, product photo backdrop"
bg = pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
bg

In [ ]:
from PIL import Image
from app_core.background import remove_background
from app_core.compose import compose_ad

cut = remove_background(Image.open("테스트사진.jpg"))
ad = compose_ad(cut, "크로플 출시 기념!", "지금 바로 3,500원에 만나요!", background=bg)
ad.save("첫_AI배경_광고.png")

small = ad.copy()
small.thumbnail((420, 420))
small

In [ ]:
from PIL import Image
from app_core.background import remove_background
from app_core.compose import compose_ad
from app_core.gen_background import generate_background

bg = generate_background("warm cozy cafe interior, wooden table, soft morning light, blurred background")
cut = remove_background(Image.open("테스트사진.jpg"))
ad = compose_ad(cut, "크로플 출시 기념!", "지금 바로 3,500원에 만나요!", background=bg)
small = ad.copy()
small.thumbnail((420, 420))
small

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 OPENAI_API_KEY를 환경변수로 올림

from app_core.prompt_builder import build_bg_prompt

p = build_bg_prompt("분식집", "신규 오픈", "활기찬 점심")
print(p)

In [ ]:
bg2 = generate_background(p)
cut2 = remove_background(Image.open("테스트사진.jpg"))
ad2 = compose_ad(cut2, "코드잇분식 오픈!", "8월 31일, 직장인 점심 특가", background=bg2)
small = ad2.copy()
small.thumbnail((420, 420))
small

In [ ]:
from app_core.photo_store import load_photo, save_photo

num = save_photo(Image.open("테스트사진.jpg"))
print(f"보관 완료 — 번호표 {num}번")

again = load_photo(num)
again.size

## 실험 기록 (2026-08-10)
- sd-turbo 2스텝, 영어 프롬프트 → 카페 배경 성공 (한글 프롬프트는 엉뚱한 그림 — 번역 부품 필요)
- AI 배경 + 누끼 + 두 줄 문구 첫 합성 성공 (첫_AI배경_광고.png)
- 개선 2호 후보: 사진 배경에서 글자 가독성 (반투명 띠 / 흰 글자+그림자)
- 개선 3호 후보: 제품 아래 그림자 (스티커 느낌 제거)
- 번역 부품: "no food" 부정어는 역효과(음식 소환) → 긍정형("empty clean surface")으로 전환
- photo_store(5호)·pipeline(6호) 검증 — generate_ad() 한 번으로 완성 광고

In [ ]:
import torch


def try_bg(prompt):
    g = torch.Generator("cuda").manual_seed(42)
    img = pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0, generator=g).images[0]
    small = img.copy()
    small.thumbnail((300, 300))
    return small


try_bg("close-up of an empty wooden tabletop, blurred snack bar interior in the background, soft warm light")

In [ ]:
try_bg("bright lively snack bar interior, close-up of an empty clean surface in the foreground, soft light")

In [ ]:
try_bg("close-up of an empty wooden tabletop, blurred snack bar interior in the background, soft warm light, low camera angle, shallow depth of field")

In [ ]:
import importlib

import app_core.pipeline

importlib.reload(app_core.pipeline)
from app_core.pipeline import generate_ad
from app_core.schema import AdBrief, CopyCandidate, Store

store = Store(id=1, user_id=1, industry="cafe", name="코드잇분식", address="서울시 마포구 연남동 1-2")
brief = AdBrief(
    goal="image", product="오므라이스", price=9900,
    situation="신규 오픈", tone="따뜻한", photo_id=1,
)
copy = CopyCandidate(headline="코드잇분식 오픈!", sub="8월 31일, 오므라이스 9,900원")

ad = generate_ad(brief, store, copy)
small = ad.copy()
small.thumbnail((420, 420))
small

In [ ]:
import importlib

import app_core.compose
import app_core.pipeline

importlib.reload(app_core.compose)
importlib.reload(app_core.pipeline)
from app_core.pipeline import generate_ad
from app_core.schema import AdBrief, CopyCandidate, Store

store = Store(id=1, user_id=1, industry="cafe", name="코드잇스터디카페", address="서울시 마포구 연남동 1-2")
brief = AdBrief(
    goal="image", product="스터디카페 이용권", price=0,
    situation="신규 오픈", tone="차분하고 집중되는", photo_id=None,
)
copy = CopyCandidate(headline="8월 31일 오픈!", sub="조용한 나만의 공간")

ad = generate_ad(brief, store, copy)
small = ad.copy()
small.thumbnail((420, 420))
small

In [ ]:
from app_core.result_store import load_result, save_result

path = save_result(ad)
print("저장됨:", path)

again = load_result(path)
again.size

In [ ]:
import importlib
import warnings

warnings.filterwarnings("ignore")

from diffusers.utils import logging as dif_log
from dotenv import load_dotenv
from PIL import Image
from transformers import logging as hf_log

hf_log.set_verbosity_error()
dif_log.set_verbosity_error()
load_dotenv()

import app_core.prompt_builder

importlib.reload(app_core.prompt_builder)
from app_core.gen_background import _load_pipe, generate_background
from app_core.prompt_builder import build_bg_prompt

_load_pipe().set_progress_bar_config(disable=True)  # 진행바 끄기

cases = [
    ("카페", "신규 오픈", "따뜻한"),
    ("분식집", "점심 특가", "활기찬"),
    ("꽃집", "봄 시즌", "화사한"),
    ("미용실", "재오픈", "세련된"),
]

imgs = []
for industry, situation, tone in cases:
    p = build_bg_prompt(industry, situation, tone)
    print(f"[{industry}] {p[:80]}")
    imgs.append(generate_background(p).resize((300, 300)))

grid = Image.new("RGB", (610, 610), (30, 30, 30))
for i, im in enumerate(imgs):
    grid.paste(im, (10 + (i % 2) * 300, 10 + (i // 2) * 300))
grid

In [ ]:
from PIL import Image, ImageDraw

from app_core.compose import _fit_font, _load_font
from app_core.gen_background import generate_background
from app_core.background import remove_background
from app_core.photo_store import load_photo


def poster_v1(product, bg, badge, shop, headline, sub, info, size=1080):
    """포스터형 v1 — 배지 / 가게명 / 제품 / 하단 정보바"""
    canvas = bg.resize((size, size)).convert("RGBA")
    d = ImageDraw.Draw(canvas, "RGBA")

    # 상단 배지 (둥근 사각 + 흰 글자)
    bf = _load_font(38)
    bw = int(bf.getlength(badge)) + 56
    d.rounded_rectangle([(size - bw) // 2, 46, (size + bw) // 2, 116], 35, fill=(150, 30, 30, 235))
    d.text((size // 2, 81), badge, font=bf, fill=(255, 245, 230), anchor="mm")

    # 가게명
    d.text((size // 2, 168), shop, font=_fit_font(shop, int(size * 0.8), 64),
           fill=(255, 255, 255), anchor="mm", stroke_width=4, stroke_fill=(50, 30, 15))

    # 제품
    if product is not None:
        p = product.crop(product.getbbox()) if product.getbbox() else product.copy()
        p.thumbnail((int(size * 0.78), int(size * 0.5)))
        canvas.alpha_composite(p, ((size - p.width) // 2, 300))

    # 하단 정보바
    d.rectangle([0, size - 250, size, size], fill=(35, 45, 38, 225))
    d.text((size // 2, size - 185), headline, font=_fit_font(headline, int(size * 0.88), 62),
           fill=(255, 255, 255), anchor="mm")
    d.text((size // 2, size - 115), sub, font=_fit_font(sub, int(size * 0.88), 40),
           fill=(240, 205, 130), anchor="mm")
    d.text((size // 2, size - 52), info, font=_fit_font(info, int(size * 0.9), 28),
           fill=(210, 215, 208), anchor="mm")

    return canvas.convert("RGB")


bg = generate_background(
    "close-up of an empty tabletop in the foreground, blurred warm korean diner interior behind"
)
product = remove_background(load_photo(1))

poster = poster_v1(
    product, bg,
    badge="GRAND OPEN",
    shop="남산왕돈까스",
    headline="8월 31일 오픈합니다",
    sub="바삭함은 기본, 푸짐함은 왕으로",
    info="서울 중구 소파로 45길 12  ·  11:00-21:00  ·  @namsan_wang",
)
small = poster.copy()
small.thumbnail((450, 450))
small

In [ ]:
def paper_background(size=1080, base=(243, 233, 210)):
    """포스터용 종이 배경 — 조용해야 정보가 읽힌다."""
    bg = Image.new("RGB", (size, size), base)
    d = ImageDraw.Draw(bg, "RGBA")
    for i in range(90):  # 가장자리로 갈수록 살짝 어둡게 (비네팅)
        d.rectangle([i, i, size - i, size - i], outline=(0, 0, 0, 3))
    return bg


CREAM, DARK, RED, GOLD = (243, 233, 210), (44, 58, 46), (150, 38, 30), (198, 154, 60)


def poster_v2(product, badge, shop, tagline, headline, sub, info, size=1080):
    canvas = paper_background(size).convert("RGBA")
    d = ImageDraw.Draw(canvas, "RGBA")

    # 태그라인 (작게)
    d.text((size // 2, 74), tagline, font=_fit_font(tagline, int(size * 0.7), 30),
           fill=(120, 105, 80), anchor="mm")

    # 가게명 (크게, 진초록)
    d.text((size // 2, 140), shop, font=_fit_font(shop, int(size * 0.82), 86),
           fill=DARK, anchor="mm")

    # 밑줄 장식
    d.line([(size * 0.32, 196), (size * 0.68, 196)], fill=GOLD, width=3)

    # 배지
    bf = _load_font(34)
    bw = int(bf.getlength(badge)) + 52
    d.rounded_rectangle([(size - bw) // 2, 216, (size + bw) // 2, 278], 31, fill=RED)
    d.text((size // 2, 247), badge, font=bf, fill=CREAM, anchor="mm")

    # 제품 — 크게, 아래쪽
    if product is not None:
        p = product.crop(product.getbbox()) if product.getbbox() else product.copy()
        p.thumbnail((int(size * 0.86), int(size * 0.46)))
        canvas.alpha_composite(p, ((size - p.width) // 2, size - 250 - p.height + 20))

    # 하단 정보바
    d.rectangle([0, size - 230, size, size], fill=DARK)
    d.text((size // 2, size - 168), headline, font=_fit_font(headline, int(size * 0.88), 60),
           fill=(255, 255, 255), anchor="mm")
    d.text((size // 2, size - 104), sub, font=_fit_font(sub, int(size * 0.88), 38),
           fill=GOLD, anchor="mm")
    d.text((size // 2, size - 46), info, font=_fit_font(info, int(size * 0.92), 26),
           fill=(200, 208, 198), anchor="mm")

    return canvas.convert("RGB")


poster = poster_v2(
    product,
    badge="GRAND OPEN",
    shop="남산왕돈까스",
    tagline="남산의 맛을 담은 추억의 한 그릇",
    headline="8월 31일 오픈합니다",
    sub="바삭함은 기본, 푸짐함은 왕으로",
    info="서울 중구 소파로 45길 12  ·  11:00-21:00  ·  @namsan_wang",
)
small = poster.copy()
small.thumbnail((450, 450))
small

In [ ]:
from PIL import Image, ImageDraw, ImageFilter


def poster_v3(product, badge, shop, tagline, headline, sub, info, event=None, size=1080):
    canvas = paper_background(size).convert("RGBA")
    d = ImageDraw.Draw(canvas, "RGBA")

    d.text((size // 2, 74), tagline, font=_fit_font(tagline, int(size * 0.7), 30),
           fill=(120, 105, 80), anchor="mm")
    d.text((size // 2, 140), shop, font=_fit_font(shop, int(size * 0.82), 86),
           fill=DARK, anchor="mm")
    d.line([(size * 0.32, 196), (size * 0.68, 196)], fill=GOLD, width=3)

    bf = _load_font(34)
    bw = int(bf.getlength(badge)) + 52
    d.rounded_rectangle([(size - bw) // 2, 216, (size + bw) // 2, 278], 31, fill=RED)
    d.text((size // 2, 247), badge, font=bf, fill=CREAM, anchor="mm")

    # 제품 뒤 스포트라이트 — 흐리게 깔아 시선을 모은다 (제품보다 먼저 그린다)
    glow = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    ImageDraw.Draw(glow).ellipse([size * 0.13, 330, size * 0.87, 830], fill=(214, 199, 170, 120))
    canvas.alpha_composite(glow.filter(ImageFilter.GaussianBlur(60)))

    if product is not None:
        p = product.crop(product.getbbox()) if product.getbbox() else product.copy()
        p.thumbnail((int(size * 0.92), int(size * 0.58)))
        canvas.alpha_composite(p, ((size - p.width) // 2, 860 - p.height))

    # 이벤트 스티커 (원형) — 레퍼런스의 OPEN EVENT 자리
    if event:
        cx, cy, r = 178, 700, 118
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=RED)
        d.ellipse([cx - r + 9, cy - r + 9, cx + r - 9, cy + r - 9], outline=CREAM, width=3)
        for i, line in enumerate(event.split("\n")):
            d.text((cx, cy - 26 + i * 40), line, font=_fit_font(line, r * 2 - 40, 36),
                   fill=CREAM, anchor="mm")

    d.rectangle([0, size - 230, size, size], fill=DARK)
    d.text((size // 2, size - 168), headline, font=_fit_font(headline, int(size * 0.88), 60),
           fill=(255, 255, 255), anchor="mm")
    d.text((size // 2, size - 104), sub, font=_fit_font(sub, int(size * 0.88), 38),
           fill=GOLD, anchor="mm")
    d.text((size // 2, size - 46), info, font=_fit_font(info, int(size * 0.92), 26),
           fill=(200, 208, 198), anchor="mm")

    return canvas.convert("RGB")


poster = poster_v3(
    product,
    badge="GRAND OPEN",
    shop="남산왕돈까스",
    tagline="남산의 맛을 담은 추억의 한 그릇",
    headline="8월 31일 오픈합니다",
    sub="바삭함은 기본, 푸짐함은 왕으로",
    info="서울 중구 소파로 45길 12  ·  11:00-21:00  ·  @namsan_wang",
    event="오픈 기념\n음료 무료",
)
small = poster.copy()
small.thumbnail((450, 450))
small

In [ ]:
import importlib

import app_core.fonts

importlib.reload(app_core.fonts)
from app_core.fonts import fit, load


def poster_v4(product, badge, shop, tagline, headline, sub, info, event=None, size=1080):
    canvas = paper_background(size).convert("RGBA")
    d = ImageDraw.Draw(canvas, "RGBA")

    d.text((size // 2, 78), tagline, font=fit(tagline, int(size * 0.7), 52, "script"),
           fill=(130, 112, 84), anchor="mm")
    d.text((size // 2, 152), shop, font=fit(shop, int(size * 0.84), 96, "display"),
           fill=DARK, anchor="mm")
    d.line([(size * 0.30, 214), (size * 0.70, 214)], fill=GOLD, width=3)

    bf = load("display", 36)
    bw = int(bf.getlength(badge)) + 56
    d.rounded_rectangle([(size - bw) // 2, 236, (size + bw) // 2, 300], 32, fill=RED)
    d.text((size // 2, 268), badge, font=bf, fill=CREAM, anchor="mm")

    glow = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    ImageDraw.Draw(glow).ellipse([size * 0.13, 340, size * 0.87, 830], fill=(214, 199, 170, 120))
    canvas.alpha_composite(glow.filter(ImageFilter.GaussianBlur(60)))

    if product is not None:
        p = product.crop(product.getbbox()) if product.getbbox() else product.copy()
        p.thumbnail((int(size * 0.92), int(size * 0.58)))
        canvas.alpha_composite(p, ((size - p.width) // 2, 860 - p.height))

    if event:
        cx, cy, r = 178, 700, 118
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=RED)
        d.ellipse([cx - r + 9, cy - r + 9, cx + r - 9, cy + r - 9], outline=CREAM, width=3)
        for i, line in enumerate(event.split("\n")):
            d.text((cx, cy - 24 + i * 42), line, font=fit(line, r * 2 - 44, 38, "display"),
                   fill=CREAM, anchor="mm")

    d.rectangle([0, size - 230, size, size], fill=DARK)
    d.text((size // 2, size - 168), headline, font=fit(headline, int(size * 0.88), 64, "display"),
           fill=(255, 255, 255), anchor="mm")
    d.text((size // 2, size - 104), sub, font=fit(sub, int(size * 0.88), 38, "body"),
           fill=GOLD, anchor="mm")
    d.text((size // 2, size - 46), info, font=fit(info, int(size * 0.92), 26, "body_light"),
           fill=(200, 208, 198), anchor="mm")

    return canvas.convert("RGB")


poster = poster_v4(
    product,
    badge="GRAND OPEN",
    shop="남산왕돈까스",
    tagline="남산의 맛을 담은 추억의 한 그릇",
    headline="8월 31일 오픈합니다",
    sub="바삭함은 기본, 푸짐함은 왕으로",
    info="서울 중구 소파로 45길 12  ·  11:00-21:00  ·  @namsan_wang",
    event="오픈 기념\n음료 무료",
)
small = poster.copy()
small.thumbnail((450, 450))
small

In [ ]:
def poster_v5(product, badge, shop, tagline, date_line, features, info, size=1080):
    """비대칭 2단 — 좌: 정보 기둥 / 우: 제품"""
    canvas = paper_background(size).convert("RGBA")
    d = ImageDraw.Draw(canvas, "RGBA")
    M = 64  # 바깥 여백

    # ── 우측: 제품 (먼저 깔고 글자를 위에 얹는다)
    if product is not None:
        glow = Image.new("RGBA", (size, size), (0, 0, 0, 0))
        ImageDraw.Draw(glow).ellipse([size * 0.42, 300, size * 1.02, 860], fill=(214, 199, 170, 130))
        canvas.alpha_composite(glow.filter(ImageFilter.GaussianBlur(70)))

        p = product.crop(product.getbbox()) if product.getbbox() else product.copy()
        p.thumbnail((int(size * 0.60), int(size * 0.52)))
        canvas.alpha_composite(p, (size - p.width - 20, 330))

    # ── 우상단 리본 배지
    if badge:
        bf = fit(badge, 210, 46, "display")
        rx0, rx1 = size - 250, size - M
        d.rectangle([rx0, 0, rx1, 190], fill=RED)
        d.polygon([(rx0, 190), (rx1, 190), ((rx0 + rx1) // 2, 232)], fill=RED)  # 아래 V컷
        for i, line in enumerate(badge.split()):
            d.text(((rx0 + rx1) // 2, 66 + i * 54), line, font=bf, fill=CREAM, anchor="mm")

    # ── 좌측 기둥
    d.text((M, 118), tagline, font=fit(tagline, 430, 46, "script"), fill=(130, 112, 84), anchor="lm")
    d.text((M, 196), shop, font=fit(shop, 470, 82, "display"), fill=DARK, anchor="lm")
    d.line([(M, 246), (M + 330, 246)], fill=GOLD, width=4)

    # 날짜 박스
    d.rounded_rectangle([M, 292, M + 400, 412], 10, outline=DARK, width=3)
    d.text((M + 200, 352), date_line, font=fit(date_line, 350, 46, "display"), fill=RED, anchor="mm")

    # 특징 3줄
    for i, (title, desc) in enumerate(features[:3]):
        y = 486 + i * 104
        d.ellipse([M, y - 16, M + 32, y + 16], fill=DARK)
        d.text((M + 52, y - 12), title, font=fit(title, 360, 34, "body"), fill=DARK, anchor="lm")
        d.text((M + 52, y + 22), desc, font=fit(desc, 380, 26, "body_light"),
               fill=(120, 110, 92), anchor="lm")
        if i < 2:
            d.line([(M, y + 52), (M + 400, y + 52)], fill=(190, 178, 152), width=1)

    # ── 하단 정보바
    d.rectangle([0, size - 132, size, size], fill=DARK)
    d.text((size // 2, size - 66), info, font=fit(info, int(size * 0.9), 28, "body_light"),
           fill=(214, 220, 210), anchor="mm")

    return canvas.convert("RGB")


poster = poster_v5(
    product,
    badge="GRAND OPEN",
    shop="남산왕돈까스",
    tagline="남산의 맛을 담은 추억의 한 그릇",
    date_line="8.31 OPEN",
    features=[
        ("100% 국내산 생등심", "신선한 고기만 사용합니다"),
        ("수제 소스", "부드럽고 진한 특제 소스"),
        ("푸짐한 양", "왕돈까스의 넉넉한 인심"),
    ],
    info="서울 중구 소파로 45길 12  ·  11:00-21:00  ·  @namsan_wang",
)
small = poster.copy()
small.thumbnail((470, 470))
small

In [ ]:
import importlib

import app_core.poster_plan

importlib.reload(app_core.poster_plan)
from app_core.poster_plan import plan_poster

plan = plan_poster(
    shop="남산왕돈까스",
    industry="일식·돈까스",
    product="왕돈까스",
    situation="신규 오픈",
    tone="",
)
plan

In [ ]:
importlib.reload(app_core.poster_plan)
from app_core.poster_plan import plan_poster

# ① 날짜·이벤트를 말한 경우 → 채워져야 함
p1 = plan_poster("남산왕돈까스", "일식·돈까스", "왕돈까스", "신규 오픈",
                 transcript="8월 31일에 오픈해요. 오픈 기념으로 음료 무료로 드리려고요")
print("① ", p1)

# ② 아무 말도 안 한 경우 → date_line·event 가 비어야 함
p2 = plan_poster("남산왕돈까스", "일식·돈까스", "왕돈까스", "신규 오픈")
print("② ", p2)

In [ ]:
from app_core.fonts import fit, load
from app_core.poster import paper_background
from app_core.poster_plan import PALETTES


def render_plan(plan, product, shop, info, size=1080):
    """기획안(plan)을 그대로 그린다 — 색도 AI가 고른 팔레트를 쓴다."""
    cream, dark, red, gold = PALETTES[plan.palette]
    canvas = paper_background(size, base=cream).convert("RGBA")
    d = ImageDraw.Draw(canvas, "RGBA")
    M = 64

    if product is not None:
        p = product.crop(product.getbbox()) if product.getbbox() else product.copy()
        p.thumbnail((int(size * 0.58), int(size * 0.50)))
        canvas.alpha_composite(p, (size - p.width - 20, 340))

    if plan.badge:
        rx0, rx1 = size - 250, size - M
        bf = fit(plan.badge, rx1 - rx0 - 32, 42, "display")
        d.rectangle([rx0, 0, rx1, 150], fill=red)
        d.polygon([(rx0, 150), (rx1, 150), ((rx0 + rx1) // 2, 192)], fill=red)
        d.text(((rx0 + rx1) // 2, 74), plan.badge, font=bf, fill=cream, anchor="mm")

    d.text((M, 118), plan.tagline, font=fit(plan.tagline, 430, 46, "script"),
           fill=dark, anchor="lm")
    d.text((M, 196), shop, font=fit(shop, 470, 82, "display"), fill=dark, anchor="lm")
    d.line([(M, 246), (M + 330, 246)], fill=gold, width=4)

    if plan.date_line:
        d.rounded_rectangle([M, 292, M + 380, 402], 10, outline=dark, width=3)
        d.text((M + 190, 347), plan.date_line, font=fit(plan.date_line, 330, 44, "display"),
               fill=red, anchor="mm")

    for i, feat in enumerate(plan.features[:3]):
        title, _, desc = feat.partition("|")
        y = 470 + i * 100
        d.ellipse([M, y - 14, M + 28, y + 14], fill=dark)
        d.text((M + 48, y - 12), title, font=fit(title, 340, 32, "body"), fill=dark, anchor="lm")
        d.text((M + 48, y + 20), desc, font=fit(desc, 360, 24, "body_light"),
               fill=(120, 110, 92), anchor="lm")

    if plan.event:
        d.rounded_rectangle([M, 790, M + 400, 880], 12, fill=red)
        d.text((M + 200, 835), plan.event, font=fit(plan.event, 350, 34, "display"),
               fill=cream, anchor="mm")

    d.rectangle([0, size - 120, size, size], fill=dark)
    d.text((size // 2, size - 60), info, font=fit(info, int(size * 0.9), 26, "body_light"),
           fill=(220, 224, 216), anchor="mm")
    return canvas.convert("RGB")


shot = render_plan(p1, product, "남산왕돈까스",
                   "서울 중구 소파로 45길 12  ·  11:00-21:00  ·  @namsan_wang")
small = shot.copy()
small.thumbnail((470, 470))
small

In [ ]:
plan_f = plan_poster(
    shop="연남 플라워",
    industry="꽃집",
    product="봄 꽃다발",
    situation="봄 시즌",
    tone="화사한",
    transcript="봄이라 꽃다발 새로 들어왔어요. 3월 한 달 동안 할인해요",
)
print(plan_f)

# 사진이 없으니 주인공 이미지를 AI 가 그린다 (빈 무대가 아니라 대상 자체)
hero = generate_background(
    "beautiful fresh spring flower bouquet, pastel colors, soft natural light, "
    "blurred bright flower shop interior behind"
).convert("RGBA")

shot = render_plan(plan_f, hero, "연남 플라워",
                   "서울 마포구 연남동 223-14  ·  10:00-20:00  ·  @yeonnam_flower")
small = shot.copy()
small.thumbnail((470, 470))
small

In [ ]:
import importlib

import app_core.palettes
import app_core.poster
import app_core.poster_plan

importlib.reload(app_core.palettes)
importlib.reload(app_core.poster)
importlib.reload(app_core.poster_plan)
from app_core.poster import generate_poster
from app_core.poster_plan import plan_poster

plan = plan_poster(
    shop="연남 플라워",
    industry="꽃집",
    product="봄 꽃다발",
    situation="봄 시즌",
    tone="화사한",
    transcript="봄이라 꽃다발 새로 들어왔어요. 3월 한 달 동안 할인해요",
)
print(plan)

shot = generate_poster(
    hero,                       # 앞에서 만든 꽃 이미지 (없으면 None 로)
    "연남 플라워",
    tagline=plan.tagline,
    badge=plan.badge,
    date_line=plan.date_line,
    features=plan.features,
    event=plan.event,
    info="서울 마포구 연남동 223-14  ·  10:00-20:00  ·  @yeonnam_flower",
    palette=plan.palette,
)
small = shot.copy()
small.thumbnail((470, 470))
small

In [ ]:
import importlib

import app_core.pipeline

importlib.reload(app_core.pipeline)
from app_core.pipeline import generate_ad
from app_core.schema import AdBrief, CopyCandidate, Store

store = Store(id=1, user_id=1, industry="cafe", name="코드잇분식", address="서울시 마포구 연남동 1-2", phone="02-333-4444")
brief = AdBrief(
    goal="image", product="오므라이스", price=9900,
    situation="신규 오픈", tone="따뜻한", photo_id=1,
    transcript=["8월 31일에 새로 열어요", "오므라이스 9,900원이에요"],
)
copy = CopyCandidate(headline="코드잇분식 오픈!", sub="오므라이스 9,900원")

simple = generate_ad(brief, store, copy, "simple")
poster = generate_ad(brief, store, copy, "poster")

both = Image.new("RGB", (simple.width * 2 + 20, simple.height), (30, 30, 30))
both.paste(simple, (0, 0))
both.paste(poster, (simple.width + 20, 0))
sm = both.copy()
sm.thumbnail((940, 470))
sm

## 실험 기록 (2026-08-12)
- 포스터형 실험 v1~v5: 사진 배경은 글자를 묻는다 → 포스터는 종이 배경(단색)으로 분리
- 레퍼런스와의 격차는 모델이 아니라 레이아웃·글꼴이었다 → 글꼴 3종(제목/본문/손글씨) 도입으로 체감 개선
- 방향 전환: 레퍼런스를 픽셀로 베끼는 게 아니라, 기획(LLM이 내용·색 결정) + 렌더(코드가 그림)로 분리
- LLM은 칸이 있으면 채운다: 할인 이벤트·없는 연도를 지어냄 → "말 안 했으면 빈 문자열" 명시 + 원문(transcript) 전달로 해결
- 색은 자유 선택 대신 검증된 팔레트 5종 중 고르게 함 (자유로 두면 촌스러워짐)
- 최종 관통: 주문서(AdBrief) 하나 → simple/poster 두 형태 (generate_ad의 style 인자)
- 남은 것: 사진 없는 포스터의 주인공(hero) 이미지 자동 생성, 구도 미세 조정

In [ ]:
import importlib

import app_core.pipeline

importlib.reload(app_core.pipeline)
from app_core.pipeline import generate_ad
from app_core.schema import AdBrief, CopyCandidate, Store

store = Store(
    id=1, user_id=1, industry="other", industry_note="꽃집",
    name="연남 플라워", address="서울시 마포구 연남동 223-14",
)
brief = AdBrief(
    goal="image", product="봄 꽃다발", price=0,
    situation="봄 시즌", tone="화사한", photo_id=None,
    transcript=["봄이라 꽃다발 새로 들어왔어요", "3월 한 달 동안 할인해요"],
)
copy = CopyCandidate(headline="봄 꽃다발 입고!")

ad = generate_ad(brief, store, copy, "poster")
small = ad.copy()
small.thumbnail((450, 450))
small